In [ ]:
'''
This method is a more stable alternative to Leave-One-Out Encoding. Instead of excluding just one observation per row, K-Fold Out-of-Fold Encoding divides the training data into K folds and computes
the target encoding for each fold without using the data from the same fold.

How K-Fold Out-of-Fold Encoding Works
1. Splits the training data into K folds.
2. For each fold:
Computes the mean target value per category using the other K-1 folds (excluding the current fold).
Each row gets an encoding based on this out-of-fold calculation.
3. For test data:
Since test data has no target values, it simply maps each category to its mean from the full training set (like in Leave-One-Out Encoding).
If a category in the test set does not appear in the training set, it uses the global mean as a fallback.

'''

'\nHow K-Fold Out-of-Fold Encoding Works\nSplits the training data into K folds.\nFor each fold:\nComputes the mean target value per category using the other K-1 folds (excluding the current fold).\nEach row gets an encoding based on this out-of-fold calculation.\nFor test data:\nSince test data has no target values, it simply maps each category to its mean from the full training set (like in Leave-One-Out Encoding).\nIf a category in the test set does not appear in the training set, it uses the global mean as a fallback.\n\n'

In [ ]:
import pandas as pd
import numpy as np
from typing import Optional, Dict, Union, List
from dataclasses import dataclass, field
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import logging
from pathlib import Path
import joblib

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class EncoderConfig:
    """Configuration class for KFoldTargetEncoder"""
    n_splits: int = 5
    shuffle: bool = True
    random_state: int = 42
    min_samples_leaf: int = 1
    smoothing_factor: float = 1.0

class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Advanced K-Fold Target Encoder with smoothing and validation.

    Features:
    - Handles multiple categorical columns
    - Implements smoothing to prevent overfitting
    - Includes validation checks
    - Provides detailed logging
    - Supports serialization

    Parameters:
    -----------
    categorical_columns : List[str]
        List of categorical column names to encode
    config : EncoderConfig
        Configuration object containing encoding parameters
    """

    def __init__(
        self,
        categorical_columns: List[str],
        target_column: str = 'target',
        config: Optional[EncoderConfig] = None
    ):
        self.categorical_columns = categorical_columns
        self.target_column = target_column
        self.config = config or EncoderConfig()
        self.encodings_: Dict[str, Dict[int, float]] = {}
        self.global_means_: Dict[str, float] = {}
        self.feature_statistics_: Dict[str, Dict[str, float]] = {}

    def _validate_input(self, X: pd.DataFrame, y: Optional[pd.Series] = None) -> None:
        """Validate input data"""
        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame")

        missing_cols = set(self.categorical_columns) - set(X.columns)
        if missing_cols:
            raise ValueError(f"Missing categorical columns: {missing_cols}")

        if y is not None and len(X) != len(y):
            raise ValueError("X and y must have the same length")

    def _compute_smoothed_target_mean(
        self,
        group_counts: pd.Series,
        group_means: pd.Series,
        global_mean: float
    ) -> pd.Series:
        """
        Compute smoothed target mean using Bayesian smoothing

        smoothed_mean = (count * mean + smoothing * global_mean) / (count + smoothing)
        """
        smoothed = (
            (group_counts * group_means + self.config.smoothing_factor * global_mean) /
            (group_counts + self.config.smoothing_factor)
        )
        return smoothed

    def fit(self, X: pd.DataFrame, y: pd.Series) -> 'KFoldTargetEncoder':
        """
        Fit the encoder using K-Fold target encoding strategy
        """
        self._validate_input(X, y)
        logger.info("Starting encoder fitting process")

        # Ensure we're working with copies to avoid modifying original data
        X = X.copy()
        y = y.copy()

        # No need to set y.name as we'll use a temporary target column name

        for column in self.categorical_columns:
            logger.debug(f"Processing column: {column}")
            self.global_means_[column] = y.mean()

            # Store feature statistics
            self.feature_statistics_[column] = {
                'unique_values': X[column].nunique(),
                'null_count': X[column].isnull().sum(),
                'most_frequent': X[column].mode().iloc[0]
            }

            # Initialize encoding dictionary
            self.encodings_[column] = {}

            # Create K-Fold splits
            kf = KFold(
                n_splits=min(self.config.n_splits, len(X)),
                shuffle=self.config.shuffle,
                random_state=self.config.random_state
            )

            # Perform K-Fold encoding
            for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
                X_train_fold = X.iloc[train_idx].copy()
                y_train_fold = y.iloc[train_idx]

                # Add target to temporary training data
                X_train_fold['_target'] = y_train_fold

                # Calculate smoothed means for the fold
                group_counts = X_train_fold[column].value_counts()
                group_means = X_train_fold.groupby(column)['_target'].mean()
                smoothed_means = self._compute_smoothed_target_mean(
                    group_counts, group_means, self.global_means_[column]
                )

                # Map values for validation fold
                for idx in val_idx:
                    current_value = X.iloc[idx][column]
                    self.encodings_[column][idx] = smoothed_means.get(
                        current_value, self.global_means_[column]
                    )

        logger.info("Encoder fitting completed successfully")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Transform categorical columns using fitted encodings
        """
        self._validate_input(X)
        X_transformed = X.copy()

        for column in self.categorical_columns:
            encoded_column = f"{column}_encoded"
            X_transformed[encoded_column] = X_transformed.index.map(
                self.encodings_.get(column, {})
            ).fillna(self.global_means_[column])

        return X_transformed

    def save(self, path: Union[str, Path]) -> None:
        """Save encoder to disk"""
        path = Path(path)
        joblib.dump(self, path)
        logger.info(f"Encoder saved to {path}")

    @classmethod
    def load(cls, path: Union[str, Path]) -> 'KFoldTargetEncoder':
        """Load encoder from disk"""
        path = Path(path)
        return joblib.load(path)

class ModelPipeline:
    """
    End-to-end modeling pipeline with preprocessing, training, and evaluation
    """
    def __init__(
        self,
        categorical_columns: List[str],
        numerical_columns: List[str],
        target_column: str,
        encoder_config: Optional[EncoderConfig] = None,
        random_state: int = 42
    ):
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns
        self.target_column = target_column
        self.encoder_config = encoder_config or EncoderConfig()
        self.random_state = random_state
        self.pipeline = self._create_pipeline()

    def _create_pipeline(self) -> Pipeline:
        """Create sklearn pipeline with preprocessing and model"""
        encoder = KFoldTargetEncoder(
            categorical_columns=self.categorical_columns,
            target_column=self.target_column,
            config=self.encoder_config
        )

        preprocessor = ColumnTransformer([
            ('num_scaler', StandardScaler(), self.numerical_columns),
            ('cat_encoder', 'passthrough', [f"{col}_encoded" for col in self.categorical_columns])
        ])

        return Pipeline([
            ('target_encoding', encoder),
            ('preprocessor', preprocessor),
            ('regressor', LinearRegression())
        ])

    def train_evaluate(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        test_size: float = 0.2
    ) -> Dict[str, float]:
        """
        Train model and evaluate performance

        Returns:
        --------
        Dict with performance metrics
        """
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state
        )

        # Train model
        logger.info("Training model...")
        self.pipeline.fit(X_train, y_train)

        # Make predictions
        y_pred = self.pipeline.predict(X_test)

        # Calculate metrics
        metrics = {
            'mse': mean_squared_error(y_test, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
            'r2': r2_score(y_test, y_pred)
        }

        logger.info("Model evaluation metrics: %s", metrics)
        return metrics

# Example usage
if __name__ == "__main__":
    # Sample data
    data = {
        'subject': ['Math', 'Science', 'Math', 'English', 'Science', 'Math', 'English', 'Science'],
        'feature1': [1, 2, 3, 4, 5, 6, 7, 8],
        'score': [80, 85, 78, 90, 88, 75, 92, 87]
    }
    df = pd.DataFrame(data)

    # Initialize and run pipeline
    pipeline = ModelPipeline(
        categorical_columns=['subject'],
        numerical_columns=['feature1'],
        target_column='score'
    )

    metrics = pipeline.train_evaluate(
        X=df[['subject', 'feature1']],
        y=df['score']
    )

2025-02-10 20:01:39,037 - __main__ - INFO - Training model...
2025-02-10 20:01:39,039 - __main__ - INFO - Starting encoder fitting process
2025-02-10 20:01:39,063 - __main__ - INFO - Encoder fitting completed successfully
2025-02-10 20:01:39,094 - __main__ - INFO - Model evaluation metrics: {'mse': 113.86103953422348, 'rmse': 10.670568847733634, 'r2': -3.5544415813689394}


In [ ]:
import pandas as pd
import numpy as np
from typing import Optional, Dict, Union, List
from dataclasses import dataclass, field
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import logging
from pathlib import Path
import joblib

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class EncoderConfig:
    """Configuration class for KFoldTargetEncoder"""
    n_splits: int = 5
    shuffle: bool = True
    random_state: int = 42
    min_samples_leaf: int = 1
    smoothing_factor: float = 1.0

class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Advanced K-Fold Target Encoder with smoothing and validation.

    Features:
    - Handles multiple categorical columns
    - Implements smoothing to prevent overfitting
    - Includes validation checks
    - Provides detailed logging
    - Supports serialization

    Parameters:
    -----------
    categorical_columns : List[str]
        List of categorical column names to encode
    config : EncoderConfig
        Configuration object containing encoding parameters
    """

    def __init__(
        self,
        categorical_columns: List[str],
        config: Optional[EncoderConfig] = None
    ):
        self.categorical_columns = categorical_columns
        self.config = config or EncoderConfig()
        self.encodings_: Dict[str, Dict[int, float]] = {}
        self.global_means_: Dict[str, float] = {}
        self.feature_statistics_: Dict[str, Dict[str, float]] = {}

    def _validate_input(self, X: pd.DataFrame, y: Optional[pd.Series] = None) -> None:
        """Validate input data"""
        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame")

        missing_cols = set(self.categorical_columns) - set(X.columns)
        if missing_cols:
            raise ValueError(f"Missing categorical columns: {missing_cols}")

        if y is not None and len(X) != len(y):
            raise ValueError("X and y must have the same length")

    def _compute_smoothed_target_mean(
        self,
        group_counts: pd.Series,
        group_means: pd.Series,
        global_mean: float
    ) -> pd.Series:
        """
        Compute smoothed target mean using Bayesian smoothing

        smoothed_mean = (count * mean + smoothing * global_mean) / (count + smoothing)
        """
        smoothed = (
            (group_counts * group_means + self.config.smoothing_factor * global_mean) /
            (group_counts + self.config.smoothing_factor)
        )
        return smoothed

    def fit(self, X: pd.DataFrame, y: pd.Series) -> 'KFoldTargetEncoder':
        """
        Fit the encoder using K-Fold target encoding strategy
        """
        self._validate_input(X, y)
        logger.info("Starting encoder fitting process")

        for column in self.categorical_columns:
            logger.debug(f"Processing column: {column}")
            self.global_means_[column] = y.mean()

            # Store feature statistics
            self.feature_statistics_[column] = {
                'unique_values': X[column].nunique(),
                'null_count': X[column].isnull().sum(),
                'most_frequent': X[column].mode().iloc[0]
            }

            # Initialize encoding dictionary
            self.encodings_[column] = {}

            # Create K-Fold splits
            kf = KFold(
                n_splits=min(self.config.n_splits, len(X)),
                shuffle=self.config.shuffle,
                random_state=self.config.random_state
            )

            # Perform K-Fold encoding
            for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train = y.iloc[train_idx]

                # Calculate smoothed means for the fold
                group_counts = X_train[column].value_counts()
                group_means = X_train.groupby(column)[y.name].mean()
                smoothed_means = self._compute_smoothed_target_mean(
                    group_counts, group_means, self.global_means_[column]
                )

                # Map values for validation fold
                for idx in val_idx:
                    current_value = X.iloc[idx][column]
                    self.encodings_[column][idx] = smoothed_means.get(
                        current_value, self.global_means_[column]
                    )

        logger.info("Encoder fitting completed successfully")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Transform categorical columns using fitted encodings
        """
        self._validate_input(X)
        X_transformed = X.copy()

        for column in self.categorical_columns:
            encoded_column = f"{column}_encoded"
            X_transformed[encoded_column] = X_transformed.index.map(
                self.encodings_.get(column, {})
            ).fillna(self.global_means_[column])

        return X_transformed

    def save(self, path: Union[str, Path]) -> None:
        """Save encoder to disk"""
        path = Path(path)
        joblib.dump(self, path)
        logger.info(f"Encoder saved to {path}")

    @classmethod
    def load(cls, path: Union[str, Path]) -> 'KFoldTargetEncoder':
        """Load encoder from disk"""
        path = Path(path)
        return joblib.load(path)

class ModelPipeline:
    """
    End-to-end modeling pipeline with preprocessing, training, and evaluation
    """
    def __init__(
        self,
        categorical_columns: List[str],
        numerical_columns: List[str],
        target_column: str,
        encoder_config: Optional[EncoderConfig] = None,
        random_state: int = 42
    ):
        self.categorical_columns = categorical_columns
        self.numerical_columns = numerical_columns
        self.target_column = target_column
        self.encoder_config = encoder_config or EncoderConfig()
        self.random_state = random_state
        self.pipeline = self._create_pipeline()

    def _create_pipeline(self) -> Pipeline:
        """Create sklearn pipeline with preprocessing and model"""
        encoder = KFoldTargetEncoder(
            categorical_columns=self.categorical_columns,
            config=self.encoder_config
        )

        preprocessor = ColumnTransformer([
            ('num_scaler', StandardScaler(), self.numerical_columns),
            ('cat_encoder', 'passthrough', [f"{col}_encoded" for col in self.categorical_columns])
        ])

        return Pipeline([
            ('target_encoding', encoder),
            ('preprocessor', preprocessor),
            ('regressor', LinearRegression())
        ])

    def train_evaluate(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        test_size: float = 0.2
    ) -> Dict[str, float]:
        """
        Train model and evaluate performance

        Returns:
        --------
        Dict with performance metrics
        """
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state
        )

        # Train model
        logger.info("Training model...")
        self.pipeline.fit(X_train, y_train)

        # Make predictions
        y_pred = self.pipeline.predict(X_test)

        # Calculate metrics
        metrics = {
            'mse': mean_squared_error(y_test, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
            'r2': r2_score(y_test, y_pred)
        }

        logger.info("Model evaluation metrics: %s", metrics)
        return metrics

# Example usage
if __name__ == "__main__":
    # Sample data
   # Sample data
    data = {
        'subject': ['Math', 'Science', 'Math', 'English', 'Science', 'Math', 'English', 'Science'],
        'feature1': [1, 2, 3, 4, 5, 6, 7, 8],
        'score': [80, 85, 78, 90, 88, 75, 92, 87]
    }
    df = pd.DataFrame(data)

    # Initialize and run pipeline
    pipeline = ModelPipeline(
        categorical_columns=['subject'],
        numerical_columns=['feature1'],
        target_column='score'  # This is now properly passed through
    )

    metrics = pipeline.train_evaluate(
        X=df[['subject', 'feature1']],
        y=df['score']
    )

2025-02-10 20:01:34,990 - __main__ - INFO - Training model...
2025-02-10 20:01:34,992 - __main__ - INFO - Starting encoder fitting process


KeyError: 'Column not found: score'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# =============================================================================
# 1️⃣ Custom K-Fold Out-of-Fold Encoder Transformer
# =============================================================================
class KFoldOutOfFoldEncoder(BaseEstimator, TransformerMixin):
    """
    Custom Transformer for K-Fold Out-of-Fold Encoding.
    Uses K-Fold splits to compute a mean target encoding, ensuring each fold’s values
    are computed without its own target value.

    Parameters:
    - column: Categorical column to encode.
    - n_splits: Number of K-Folds (default = 5).
    """
    def __init__(self, column, n_splits=5):
        self.column = column
        self.n_splits = n_splits
        self.global_mean_ = None
        self.mapping_ = {}

    def fit(self, X, y):
        self.global_mean_ = y.mean()
        X_temp = X.copy()
        X_temp['_target'] = y
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)

        self.mapping_ = {}
        for train_idx, val_idx in kf.split(X_temp):
            X_train, X_val = X_temp.iloc[train_idx], X_temp.iloc[val_idx]
            fold_means = X_train.groupby(self.column)['_target'].mean().to_dict()
            for idx in val_idx:
                self.mapping_[idx] = fold_means.get(X_temp.loc[idx, self.column], self.global_mean_)
        return self

    def transform(self, X):
        X_temp = X.copy()
        X_temp[self.column + '_kfold'] = X_temp.index.map(self.mapping_).fillna(self.global_mean_)
        return X_temp

# =============================================================================
# 2️⃣ Sample Data Preparation
# =============================================================================
data = {
    'subject': ['Math', 'Science', 'Math', 'English', 'Science', 'Math', 'English', 'Science'],
    'feature1': [1, 2, 3, 4, 5, 6, 7, 8],  # Numeric feature
    'score': [80, 85, 78, 90, 88, 75, 92, 87]  # Target variable (to predict)
}
df = pd.DataFrame(data)

# Separate features and target.
X = df[['subject', 'feature1']]
y = df['score']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# =============================================================================
# 3️⃣ Building the Pipeline Using K-Fold Out-of-Fold Encoding
# =============================================================================
pipeline = Pipeline([
    ('target_encoding', KFoldOutOfFoldEncoder(column='subject', n_splits=5)),  # Feature Engineering
    ('column_selector', ColumnTransformer([
        ('num_features', 'passthrough', ['feature1']),  # Keep numeric features as is
        ('encoded', 'passthrough', ['subject_kfold'])   # Use transformed categorical feature
    ])),
    ('regressor', LinearRegression())  # Model Training
])

# =============================================================================
# 4️⃣ Wrap the Pipeline in GridSearchCV for Hyperparameter Tuning
# =============================================================================
param_grid = {'regressor__fit_intercept': [True, False]}  # Hyperparameters to tune

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

# =============================================================================
# 5️⃣ Make Predictions and Evaluate the Model
# =============================================================================
y_pred = grid.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("Best Model Hyperparameters:", grid.best_params_)
print("Test Set Predictions:", y_pred)
print("Mean Squared Error on Test Set:", mse)


ValueError: 
All the 6 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
6 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\cheng\Workspace\Paramount\.venv\lib\site-packages\sklearn\model_selection\_validation.py", line 729, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\cheng\Workspace\Paramount\.venv\lib\site-packages\sklearn\base.py", line 1152, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\cheng\Workspace\Paramount\.venv\lib\site-packages\sklearn\pipeline.py", line 423, in fit
    Xt = self._fit(X, y, **fit_params_steps)
  File "c:\Users\cheng\Workspace\Paramount\.venv\lib\site-packages\sklearn\pipeline.py", line 377, in _fit
    X, fitted_transformer = fit_transform_one_ca

In [3]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

# =============================================================================
# Custom K-Fold Out-of-Fold Encoder Transformer
# =============================================================================
class KFoldOutOfFoldEncoder(BaseEstimator, TransformerMixin):
    """
    A custom transformer that applies K-Fold target encoding to a categorical column.
    Each fold's encoding is computed using the target values from the other K-1 folds.

    For test data, it maps each category to the full-dataset mean from the training set.
    """
    def __init__(self, column, n_splits=5):
        self.column = column
        self.n_splits = n_splits
        self.global_mean_ = None
        self.mapping_ = {}

    def fit(self, X, y):
        self.global_mean_ = y.mean()
        X_temp = X.copy()
        X_temp['_target'] = y
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)

        # Store mappings for each index
        self.mapping_ = {}
        for train_idx, val_idx in kf.split(X_temp):
            X_train, X_val = X_temp.iloc[train_idx], X_temp.iloc[val_idx]
            fold_means = X_train.groupby(self.column)['_target'].mean().to_dict()

            # Assign encoding from the other folds
            for idx in val_idx:
                self.mapping_[idx] = fold_means.get(X_temp.loc[idx, self.column], self.global_mean_)

        return self

    def transform(self, X):
        X_temp = X.copy()
        # Map each row to the precomputed encodings from training



In [4]:
from sklearn.model_selection import KFold

class KFoldOutOfFoldEncoder:
    def __init__(self, column, n_splits=5):
        self.column = column
        self.n_splits = n_splits
        self.global_mean_ = None
        self.mapping_ = {}

    def fit(self, X, y):
        self.global_mean_ = y.mean()
        X_temp = X.copy()
        X_temp['_target'] = y
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)

        self.mapping_ = {}
        for train_idx, val_idx in kf.split(X_temp):
            X_train, X_val = X_temp.iloc[train_idx], X_temp.iloc[val_idx]
            fold_means = X_train.groupby(self.column)['_target'].mean().to_dict()
            for idx in val_idx:
                self.mapping_[idx] = fold_means.get(X_temp.loc[idx, self.column], self.global_mean_)

    def transform(self, X):
        X_temp = X.copy()
        X_temp[self.column + '_kfold'] = X_temp.index.map(self.mapping_).fillna(self.global_mean_)
        return X_temp
